# 하이퍼파라미터 튜닝

In [ ]:
import numpy as np 
import pandas as pd from sklearn.model_selection 
import train_test_split 

wine = pd.read_csv('https://bit.ly/wine_csv_data') 
data = wine[['alcohol', 'sugar', 'pH']].to_numpy() 
target = wine['class'].to_numpy() 
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint, uniform
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_cv(model, X, y, cv=cv):
    scores = cross_validate(model, X, y, cv=cv, return_train_score=True, n_jobs=-1)
    return np.mean(scores['train_score']), np.mean(scores['test_score'])

def summarize_cv(name, model):
    tr, te = eval_cv(model, train_input, train_target, cv)
    return [name, tr, te, tr - te]

def print_table(title, df):
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))
    print(df.to_string(index=False))

# 1) 베이스라인 모델들 (스케일 필요한 모델은 파이프라인으로)
baselines = []

pipe_lr = Pipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(max_iter=2000, solver='lbfgs'))])
baselines.append(summarize_cv("LogisticRegression", pipe_lr))

pipe_svc = Pipeline([('scaler', StandardScaler()),
                     ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))])
baselines.append(summarize_cv("SVM(RBF)", pipe_svc))

pipe_knn = Pipeline([('scaler', StandardScaler()),
                     ('clf', KNeighborsClassifier(n_neighbors=5))])
baselines.append(summarize_cv("KNN(k=5)", pipe_knn))

dt = DecisionTreeClassifier(random_state=42)
baselines.append(summarize_cv("DecisionTree", dt))

rf = RandomForestClassifier(n_estimators=200, random_state=42)
baselines.append(summarize_cv("RandomForest(200)", rf))

gb = GradientBoostingClassifier(random_state=42)
baselines.append(summarize_cv("GradientBoosting", gb))

df_base = pd.DataFrame(baselines, columns=["Model","CV_Train","CV_Valid","Gap(Train-Valid)"])
df_base = df_base.sort_values("CV_Valid", ascending=False).reset_index(drop=True)
print_table("Baselines (5-fold CV)", df_base)

# 2) 하이퍼파라미터 튜닝
# 2-1) GridSearchCV: LR / SVM / KNN / DecisionTree
grid_lr = {
    'scaler': [StandardScaler()],
    'clf__C': [0.1, 1.0, 10.0, 100.0],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs'],
    'clf__max_iter': [2000]
}
gs_lr = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    grid_lr, cv=cv, n_jobs=-1, return_train_score=True
).fit(train_input, train_target)

grid_svc = {
    'clf__C': [0.1, 1.0, 3.0, 10.0],
    'clf__gamma': ['scale', 0.01, 0.05, 0.1]
}
gs_svc = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf'))]),
    grid_svc, cv=cv, n_jobs=-1, return_train_score=True
).fit(train_input, train_target)

grid_knn = {
    'clf__n_neighbors': [3,5,7,9,11],
    'clf__weights': ['uniform','distance'],
    'clf__p': [1,2],  # 1: Manhattan, 2: Euclidean
}
gs_knn = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]),
    grid_knn, cv=cv, n_jobs=-1, return_train_score=True
).fit(train_input, train_target)

grid_dt = {
    'max_depth': [None, 2, 3, 4, 5, 8, 12],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 3]
}
gs_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    grid_dt, cv=cv, n_jobs=-1, return_train_score=True
).fit(train_input, train_target)

# 2-2) RandomizedSearchCV: RandomForest / GradientBoosting
dist_rf = {
    'n_estimators': randint(100, 600),
    'max_depth': [None, 3, 5, 8, 12],
    'min_samples_split': randint(2, 12),
    'min_samples_leaf': randint(1, 6),
    'max_features': ['sqrt', 'log2', None]
}
rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    dist_rf, n_iter=30, cv=cv, n_jobs=-1, random_state=42, return_train_score=True
).fit(train_input, train_target)

dist_gb = {
    'n_estimators': randint(100, 800),
    'learning_rate': uniform(0.03, 0.2),
    'max_depth': randint(1, 4)
}
rs_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    dist_gb, n_iter=30, cv=cv, n_jobs=-1, random_state=42, return_train_score=True
).fit(train_input, train_target)

# 3) 튜닝된 모델들의 CV 성능 비교
tuned_rows = []
for name, est in [
    ("LogReg (tuned)", gs_lr.best_estimator_),
    ("SVM (tuned)", gs_svc.best_estimator_),
    ("KNN (tuned)", gs_knn.best_estimator_),
    ("DecisionTree (tuned)", gs_dt.best_estimator_),
    ("RandomForest (tuned)", rs_rf.best_estimator_),
    ("GradientBoosting (tuned)", rs_gb.best_estimator_),
]:
    tr, te = eval_cv(est, train_input, train_target, cv)
    tuned_rows.append([name, tr, te, tr - te])

df_tuned = pd.DataFrame(tuned_rows, columns=["Model","CV_Train","CV_Valid","Gap(Train-Valid)"])
df_tuned = df_tuned.sort_values("CV_Valid", ascending=False).reset_index(drop=True)
print_table("Tuned Models (5-fold CV)", df_tuned)

# 4) 최종 테스트셋 평가 (상위 몇 개만)
candidates = {
    "SVM (tuned)": gs_svc.best_estimator_,
    "RandomForest (tuned)": rs_rf.best_estimator_,
    "GradientBoosting (tuned)": rs_gb.best_estimator_,
    "LogReg (tuned)": gs_lr.best_estimator_,
}
print("\n===== Test Set Evaluation =====")
for name, model in candidates.items():
    model.fit(train_input, train_target)
    pred = model.predict(test_input)
    acc = accuracy_score(test_target, pred)
    print(f"\n[{name}] Test Accuracy: {acc:.4f}")
    print(classification_report(test_target, pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(test_target, pred))

# 5) 최적 하이퍼파라미터 요약
def show_best(tag, search):
    print(f"\n### {tag}")
    print("Best CV Score:", search.best_score_)
    print("Best Params:", search.best_params_)

show_best("LogReg (GridSearch)", gs_lr)
show_best("SVM (GridSearch)", gs_svc)
show_best("KNN (GridSearch)", gs_knn)
show_best("DecisionTree (GridSearch)", gs_dt)
show_best("RandomForest (RandomizedSearch)", rs_rf)
show_best("GradientBoosting (RandomizedSearch)", rs_gb)